In [1]:
%reload_ext autoreload
%autoreload 2

# Data Alignment Validation

Validates that all 5 supplemental datasets align correctly to the cdnnba spine via `CDNNBAMemoPL.load_all()` and the `aligned_*` memo methods.

# Imports

In [2]:
from kret_notebook import *
from kret_polars._core.polars_nb_imports import *

from nba_timeout_impact.nb_imports import *

Loaded environment variables from /home/Akseldkw/coding/nba/NBA-Timeout-Impact/.env
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0940 seconds


# Load All Datasets

In [3]:
memo = CDNNBAMemoPL.load_all()
spine = memo.cdnnba
print(f"Spine: {spine.height:,} rows, {len(spine.columns)} cols")
print(f"Boxscores: {memo.boxscores.height:,} rows")
print(f"Player Advanced Stats: {memo.player_advanced_stats.height:,} rows")
print(f"Player Season Stats: {memo.player_season_stats.height:,} rows")
print(f"Rotations: {memo.rotations.height:,} rows")
print(f"Stints: {memo.stints.height:,} rows")

Validating cdnnba data (Polars)...
  Passed (4,168,786 rows, 74 cols, 1 warnings).
Validating boxscores data (Polars)...
  Passed (217,190 rows, 36 cols).
Validating player_advanced_stats data (Polars)...
  Passed (3,400 rows, 119 cols).
Validating player_season_stats data (Polars)...
  Passed (5,174 rows, 31 cols).
Validating rotations data (Polars)...
  Passed (1,905,583 rows, 16 cols, 1 warnings).
Validating stints data (Polars)...
  Passed (2,145,106 rows, 14 cols, 1 warnings).
Spine: 4,168,786 rows, 74 cols
Boxscores: 217,190 rows
Player Advanced Stats: 3,400 rows
Player Season Stats: 5,174 rows
Rotations: 1,905,583 rows
Stints: 2,145,106 rows


In [4]:
n_player_rows = (spine["personId"] > 0).sum()
n_zero_pid = (spine["personId"] == 0).sum()
print(f"Spine rows with personId > 0: {n_player_rows:,} ({n_player_rows/spine.height:.1%})")
print(f"Spine rows with personId == 0: {n_zero_pid:,} ({n_zero_pid/spine.height:.1%})")
print("(personId==0 rows are expected to have nulls in all aligned datasets)")

Spine rows with personId > 0: 3,849,123 (92.3%)
Spine rows with personId == 0: 319,663 (7.7%)
(personId==0 rows are expected to have nulls in all aligned datasets)


# 1. ptr_boxscores

Row index into `boxscores` for each spine row, joined on `(gameId, personId)`. Materialize via `memo.boxscores[col].gather(ptr)`.

In [5]:
ptr_box = memo.ptr_boxscores
assert len(ptr_box) == spine.height, f"Length mismatch: {len(ptr_box)} vs {spine.height}"
assert ptr_box.dtype == pl.UInt32

nonnull = ptr_box.drop_nulls().len()
null_rate = ptr_box.null_count() / spine.height
print(f"ptr_boxscores: {len(ptr_box):,} entries, dtype={ptr_box.dtype}")
print(f"  non-null: {nonnull:,} ({1-null_rate:.1%})")
print(f"  null:     {ptr_box.null_count():,} ({null_rate:.1%})")

# Materialize a column to verify
points = memo.boxscores["points"].gather(ptr_box.fill_null(0)).set(~ptr_box.is_not_null(), None)
print(f"  Materialized 'points': non-null={points.drop_nulls().len():,}")
points.head(10)

Calculating ptr_boxscores
ptr_boxscores: 4,168,786 entries, dtype=UInt32
  non-null: 3,840,903 (92.1%)
  null:     327,883 (7.9%)
  Materialized 'points': non-null=3,840,903


points
i64
null
null
4
26
20
20
26
19
22


In [6]:
# Spot check: pick a player+game with a valid pointer and verify it resolves correctly
spine_df = pl.DataFrame._from_pydf(spine._df).with_row_index("_idx")
ptr_box_df = ptr_box.to_frame("_ptr").with_row_index("_idx")
sample = (
    spine_df
    .filter((pl.col("personId") > 0) & (pl.col("actionType") == "2pt"))
    .join(ptr_box_df, on="_idx")
    .filter(pl.col("_ptr").is_not_null())
    .head(1)
)
idx = int(sample["_idx"][0])
gid, pid = sample["gameId"][0], sample["personId"][0]
player_name = sample["playerName"][0]

# Resolve pointer
ptr_val = int(ptr_box[idx])
box_row = memo.boxscores.row(ptr_val, named=True)
print(f"Spot check: {player_name} in game {gid}")
print(f"  spine row {idx} -> ptr={ptr_val} -> boxscores row")
print(f"  Boxscore points: {box_row['points']}, FGM/FGA: {box_row['fieldGoalsMade']}/{box_row['fieldGoalsAttempted']}")

# All spine rows for same player+game should resolve to the same boxscore row
mask = (spine["gameId"] == gid) & (spine["personId"] == pid)
ptrs_for_player = ptr_box.filter(mask)
assert ptrs_for_player.n_unique() == 1, "All spine rows for same player+game should point to same boxscore row"
print(f"  All {mask.sum()} spine rows point to same boxscore row: True")

Spot check: Irving in game 22000001
  spine row 6 -> ptr=121963 -> boxscores row
  Boxscore points: 26, FGM/FGA: 10/16
  All 29 spine rows point to same boxscore row: True


# 2. ptr_player_advanced_stats

Row index into `player_advanced_stats`, joined on `(personId=PLAYER_ID, season=season_int)`.

In [7]:
ptr_pas = memo.ptr_player_advanced_stats
assert len(ptr_pas) == spine.height
assert ptr_pas.dtype == pl.UInt32

nonnull = ptr_pas.drop_nulls().len()
null_rate = ptr_pas.null_count() / spine.height
print(f"ptr_player_advanced_stats: {len(ptr_pas):,} entries")
print(f"  non-null: {nonnull:,} ({1-null_rate:.1%})")
print(f"  null:     {ptr_pas.null_count():,} ({null_rate:.1%})")

# Materialize NET_RATING
net_rating = memo.player_advanced_stats["NET_RATING"].gather(ptr_pas.fill_null(0)).set(~ptr_pas.is_not_null(), None)
print(f"  Materialized 'NET_RATING': non-null={net_rating.drop_nulls().len():,}")
net_rating.head(10)

Calculating ptr_player_advanced_stats
ptr_player_advanced_stats: 4,168,786 entries
  non-null: 3,848,196 (92.3%)
  null:     320,590 (7.7%)
  Materialized 'NET_RATING': non-null=3,848,196


NET_RATING
f64
null
null
-1.9
5.6
4.6
4.6
5.6
-8.8
10.7


In [8]:
# Spot check: verify a player's pointer is consistent across a season
sample = (
    pl.DataFrame._from_pydf(spine._df)
    .filter(pl.col("personId") > 0)
    .select("personId", "season", "playerName")
    .unique()
    .head(1)
)
pid, szn, pname = sample["personId"][0], sample["season"][0], sample["playerName"][0]

mask = (spine["personId"] == pid) & (spine["season"] == szn)
ptrs = ptr_pas.filter(mask)
print(f"Spot check: {pname} (pid={pid}) in season {szn}")
print(f"  All {mask.sum()} spine rows point to same row: {ptrs.n_unique() == 1}")
assert ptrs.n_unique() == 1

# Resolve and show stats
row_idx = int(ptrs[0])
print(f"  ptr={row_idx} -> NET_RATING={memo.player_advanced_stats['NET_RATING'][row_idx]}, "
      f"USG_PCT={memo.player_advanced_stats['USG_PCT'][row_idx]}")

Spot check: McClung (pid=1630644) in season 2025
  All 68 spine rows point to same row: True
  ptr=2639 -> NET_RATING=-13.3, USG_PCT=0.224


# 3. ptr_player_season_stats

Row index into `player_season_stats` (TOT-deduped), joined on `(personId, season, season_type)`.

In [9]:
ptr_pss = memo.ptr_player_season_stats
assert len(ptr_pss) == spine.height
assert ptr_pss.dtype == pl.UInt32

nonnull = ptr_pss.drop_nulls().len()
null_rate = ptr_pss.null_count() / spine.height
print(f"ptr_player_season_stats: {len(ptr_pss):,} entries")
print(f"  non-null: {nonnull:,} ({1-null_rate:.1%})")
print(f"  null:     {ptr_pss.null_count():,} ({null_rate:.1%})")

# Materialize PTS
pts = memo.player_season_stats["PTS"].gather(ptr_pss.fill_null(0)).set(~ptr_pss.is_not_null(), None)
print(f"  Materialized 'PTS': non-null={pts.drop_nulls().len():,}")
pts.head(10)

Calculating ptr_player_season_stats
ptr_player_season_stats: 4,168,786 entries
  non-null: 3,844,280 (92.2%)
  null:     324,506 (7.8%)
  Materialized 'PTS': non-null=3,844,280


PTS
i64
null
null
426
1451
2015
2015
1451
448
943


In [10]:
# Verify no fan-out: ptr length must equal spine length
assert len(ptr_pss) == spine.height, "Fan-out detected — dedup logic may be broken"

# Verify TOT preference: find a traded player present in both spine and player_season_stats
pss_df = pl.DataFrame._from_pydf(memo.player_season_stats._df)
tot_players = pss_df.filter(pl.col("TEAM_ABBREVIATION") == "TOT")
spine_players = pl.DataFrame._from_pydf(spine._df).select("personId", "season").unique()
candidates = (
    tot_players
    .rename({"PLAYER_ID": "personId", "season_int": "season"})
    .join(spine_players, on=["personId", "season"], how="inner")
)
if candidates.height > 0:
    pid_traded = candidates["personId"][0]
    szn_traded = candidates["season"][0]
    mask = (spine["personId"] == pid_traded) & (spine["season"] == szn_traded)
    ptrs_traded = ptr_pss.filter(mask).drop_nulls()
    resolved_team = memo.player_season_stats["TEAM_ABBREVIATION"][int(ptrs_traded[0])]
    print(f"Traded player check (pid={pid_traded}, season={szn_traded}):")
    print(f"  ptr resolves to TEAM_ABBREVIATION: {resolved_team}")
    assert resolved_team == "TOT", f"Expected TOT, got {resolved_team}"
    print("  TOT preference confirmed")
else:
    print("No traded players found in spine — skipping TOT check")

No traded players found in spine — skipping TOT check


# 4. ptr_stints

Row index into `stints` via `join_asof` on `(gameId, personId)` with `in_game_seconds <= game_seconds_elapsed < out_game_seconds`.

In [11]:
ptr_st = memo.ptr_stints
assert len(ptr_st) == spine.height
assert ptr_st.dtype == pl.UInt32

nonnull = ptr_st.drop_nulls().len()
null_rate = ptr_st.null_count() / spine.height
print(f"ptr_stints: {len(ptr_st):,} entries")
print(f"  non-null: {nonnull:,} ({1-null_rate:.1%})")
print(f"  null:     {ptr_st.null_count():,} ({null_rate:.1%})")

# Materialize stint_id
stint_id = memo.stints["stint_id"].gather(ptr_st.fill_null(0)).set(~ptr_st.is_not_null(), None)
print(f"  Materialized 'stint_id': non-null={stint_id.drop_nulls().len():,}")
stint_id.head(10)

Calculating ptr_stints
ptr_stints: 4,168,786 entries
  non-null: 3,370,794 (80.9%)
  null:     797,992 (19.1%)
  Materialized 'stint_id': non-null=3,370,794


/home/Akseldkw/coding/nba/NBA-Timeout-Impact/nba_timeout_impact/datasets/memo_cdnnba_pl.py:150: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


stint_id
u32
null
null
1
1
1
1
1
1
1


In [12]:
# Spot check: verify stint pointer resolves to correct time range
spine_df = pl.DataFrame._from_pydf(spine._df).with_row_index("_idx")
ptr_st_df = ptr_st.to_frame("_ptr").with_row_index("_idx")
sample_play = (
    spine_df
    .filter((pl.col("personId") > 0) & (pl.col("actionType") == "2pt"))
    .join(ptr_st_df, on="_idx")
    .filter(pl.col("_ptr").is_not_null())
    .head(1)
)
idx = int(sample_play["_idx"][0])
gse = sample_play["game_seconds_elapsed"][0]
pname = sample_play["playerName"][0]

ptr_val = int(ptr_st[idx])
stint_row = memo.stints.row(ptr_val, named=True)
print(f"Spot check: {pname} at game_seconds_elapsed={gse}")
print(f"  ptr={ptr_val} -> stints row: stint_id={stint_row['stint_id']}, "
      f"in={stint_row['in_game_seconds']:.1f}s, out={stint_row['out_game_seconds']:.1f}s")

# Verify the play falls within the stint's time range
assert stint_row["in_game_seconds"] <= gse < stint_row["out_game_seconds"], \
    f"game_seconds_elapsed={gse} not in [{stint_row['in_game_seconds']}, {stint_row['out_game_seconds']})"
print("  Time range check passed")

Spot check: Irving at game_seconds_elapsed=38.0
  ptr=1490270 -> stints row: stint_id=1, in=0.0s, out=571.0s
  Time range check passed


# 5. ptr_rotations

Row index into `rotations` via `join_asof` on `(gameId, personId)` with `IN_TIME_REAL/10 <= game_seconds_elapsed < OUT_TIME_REAL/10`.

In [13]:
ptr_rot = memo.ptr_rotations
assert len(ptr_rot) == spine.height
assert ptr_rot.dtype == pl.UInt32

nonnull = ptr_rot.drop_nulls().len()
null_rate = ptr_rot.null_count() / spine.height
print(f"ptr_rotations: {len(ptr_rot):,} entries")
print(f"  non-null: {nonnull:,} ({1-null_rate:.1%})")
print(f"  null:     {ptr_rot.null_count():,} ({null_rate:.1%})")

# Materialize location
location = memo.rotations["location"].gather(ptr_rot.fill_null(0)).set(~ptr_rot.is_not_null(), None)
print(f"  Materialized 'location': non-null={location.drop_nulls().len():,}")
location.head(10)

Calculating ptr_rotations


/home/Akseldkw/coding/nba/NBA-Timeout-Impact/nba_timeout_impact/datasets/memo_cdnnba_pl.py:190: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


ptr_rotations: 4,168,786 entries
  non-null: 3,370,866 (80.9%)
  null:     797,920 (19.1%)
  Materialized 'location': non-null=3,370,866


location
str
null
null
"""home"""
"""home"""
"""away"""
"""away"""
"""home"""
"""away"""
"""home"""


In [14]:
# Cross-check: stints and rotations pointers should agree on coverage
# Both should be non-null for the same spine rows (within overlapping seasons)
stints_nonnull = ptr_st.is_not_null()
rot_nonnull = ptr_rot.is_not_null()
both_nonnull = (stints_nonnull & rot_nonnull).sum()
stints_only = (stints_nonnull & ~rot_nonnull).sum()
rot_only = (~stints_nonnull & rot_nonnull).sum()
print(f"Stints vs Rotations pointer coverage:")
print(f"  Both non-null:  {both_nonnull:,}")
print(f"  Stints only:    {stints_only:,}")
print(f"  Rotations only: {rot_only:,}")
print(f"  Agreement rate: {both_nonnull / max(stints_nonnull.sum(), rot_nonnull.sum()):.1%}")

Stints vs Rotations pointer coverage:
  Both non-null:  3,370,779
  Stints only:    15
  Rotations only: 87
  Agreement rate: 100.0%


# Summary

In [15]:
ptrs = {
    "ptr_boxscores": ptr_box,
    "ptr_player_advanced_stats": ptr_pas,
    "ptr_player_season_stats": ptr_pss,
    "ptr_stints": ptr_st,
    "ptr_rotations": ptr_rot,
}

print(f"{'Pointer':<32} {'Length':>12} {'Non-null':>12} {'Coverage':>10} {'dtype':>8}")
print("-" * 78)
for name, ptr in ptrs.items():
    nonnull = ptr.drop_nulls().len()
    pct = nonnull / spine.height
    ok = "OK" if len(ptr) == spine.height else "MISMATCH"
    print(f"{name:<32} {len(ptr):>12,} {nonnull:>12,} {pct:>9.1%}  {ptr.dtype}  {ok}")

print(f"\nAll pointers match spine length ({spine.height:,} rows): "
      f"{all(len(p) == spine.height for p in ptrs.values())}")
print(f"Memory: 5 x UInt32 Series = ~{5 * spine.height * 4 / 1e6:.0f} MB (vs full materialized DataFrames)")

Pointer                                Length     Non-null   Coverage    dtype
------------------------------------------------------------------------------
ptr_boxscores                       4,168,786    3,840,903     92.1%  UInt32  OK
ptr_player_advanced_stats           4,168,786    3,848,196     92.3%  UInt32  OK
ptr_player_season_stats             4,168,786    3,844,280     92.2%  UInt32  OK
ptr_stints                          4,168,786    3,370,794     80.9%  UInt32  OK
ptr_rotations                       4,168,786    3,370,866     80.9%  UInt32  OK

All pointers match spine length (4,168,786 rows): True
Memory: 5 x UInt32 Series = ~83 MB (vs full materialized DataFrames)
